In [37]:
#Importing necessary Libraries
import pandas as pd
import numpy as np

In [38]:
# Loading the dataset
file_path= "../Data/Loan_acc_prediction_FCS_exercise.xlsx"
loan_df = pd.read_excel(file_path, sheet_name="Loan Demographics")
comm_df = pd.read_excel(file_path, sheet_name="Communication Activity")

In [39]:
# Temporal Validation- Check for unrealistic dates

date_columns = [
    'birthday',
    'last_pmt_date',
    'lastNoticeSent',
    'chargeoff_date'
]

display(loan_df[date_columns].agg(['min', 'max']))

,birthday,last_pmt_date,lastNoticeSent,chargeoff_date
min,1900-01-01,2022-06-19,1900-01-01,1999-09-20
max,2024-11-04,2025-08-31,2025-09-26,2025-05-31


In [40]:
# Validating 'last_pmt_date','lastNoticeSent','chargeoff_date'

minimum_valid_date = pd.Timestamp('2000-01-01')

for col in [
    'last_pmt_date',
    'lastNoticeSent',
    'chargeoff_date'
]:
    
    loan_df.loc[
        loan_df[col] < minimum_valid_date,
        col
    ] = pd.NaT

In [41]:
# Validating 'birthday'

# Create Customer Age

loan_df['customer_age'] = (
    pd.Timestamp.today().year -
    loan_df['birthday'].dt.year
) 

#Inspect unrealistic ages

invalid_age_df = loan_df[
    (loan_df['customer_age'] < 18) |
    (loan_df['customer_age'] > 100)
]

print("Invalid Age Count:")
print(invalid_age_df.shape[0])

display(
    invalid_age_df[
        ['birthday', 'customer_age']
    ].head()
)

#Fix Invalid Ages

loan_df.loc[
    (loan_df['customer_age'] < 18) |
    (loan_df['customer_age'] > 100),
    'customer_age'
] = np.nan

Invalid Age Count:
15193


,birthday,customer_age
60,1900-01-01,126
67,1900-01-01,126
99,1900-01-01,126
139,1900-01-01,126
160,1900-01-01,126


In [42]:
# Check for future dates

future_payment_dates = loan_df[
    loan_df['last_pmt_date'] > pd.Timestamp.today()
]

print("Future Payment Dates:")
print(future_payment_dates.shape[0])

Future Payment Dates:
0


In [43]:
# Check impossible temporal sequence

impossible_sequence_df = loan_df[
    loan_df['last_pmt_date'] < loan_df['birthday']
]

print("Impossible Temporal Sequences:")
print(impossible_sequence_df.shape[0])

Impossible Temporal Sequences:
0


In [48]:
#  Handle missing last payment dates

reference_date = loan_df['last_pmt_date'].max()

loan_df['days_since_last_payment'] = (
    reference_date - loan_df['last_pmt_date']
).dt.days

# Missing means no observed historical payment

loan_df['days_since_last_payment'] = (
    loan_df['days_since_last_payment']
    .fillna(9999) #Why 9999?-missing here means “very long ago / never paid” which is behaviorally meaningful tree-based models handle this very well
)


#  Handle missing state values

loan_df['state'] = (
    loan_df['state']
    .fillna('Unknown')
)

In [46]:
# Saving the validated dataframe
loan_df.to_excel(
    '../data/validated_loan_df.xlsx',
    index=False
)

comm_df.to_excel(
    '../data/validated_comm_df.xlsx',
    index=False
)

print("Validated Excel files saved successfully.")

Validated Excel files saved successfully.
